# STE RAGU Date Cutoff Diagnostic

Systematically tests each source table and query in the STE RAGU postintegration
pipeline to identify which component stops producing data after the week of
August 9, 2026.

**Pipeline under test:**
- `ste_ragu_temptables.txt` (temp tables on shared connection)
- `ste_ragu_ula.txt` -> `ula_df_total`
- `ste_ragu_recovery.txt` -> `new_recovery`
- `ste_ragu_weekly.txt` -> `ste_weekly_raw`

In [11]:
# =============================================================================
# CELL 1: CONNECTION, IMPORTS, HELPERS
# =============================================================================

import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os
import datetime as dt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 200)

DSN = 'Redshift_prod_new'
CUTOFF_REFERENCE = '2026-08-09'

conn = pyodbc.connect(f'DSN={DSN}')
print(f'Redshift connection OK  (DSN={DSN})')


def run_diag(query, connection=None):
    """Execute a diagnostic SQL query and return a DataFrame."""
    c = connection or conn
    warnings.filterwarnings('ignore', category=UserWarning)
    df = pd.read_sql_query(sql=query, con=c)
    warnings.filterwarnings('default', category=UserWarning)
    return df


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


print(f'Reference cutoff date: {CUTOFF_REFERENCE}')
print(f'Diagnostic run at: {dt.datetime.now():%Y-%m-%d %H:%M}')

Redshift connection OK  (DSN=Redshift_prod_new)
Reference cutoff date: 2026-08-09
Diagnostic run at: 2026-09-03 22:48


In [12]:
# =============================================================================
# CELL 2: CHECK EACH SOURCE TABLE MAX DATE (weekly counts >= 2026-07-01)
# =============================================================================

table_checks = {
    'edwnpi.los_deal_current_fact (book_date)': """
        SELECT date_trunc('week', book_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact
        WHERE book_date >= '2026-07-01'
          AND data_source_name != 'SPARTAN'
        GROUP BY 1 ORDER BY 1
    """,
    'edwnpi.los_deal_current_fact (app_received_dtm)': """
        SELECT date_trunc('week', application_received_dtm)::date AS week_start,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact
        WHERE application_received_dtm >= '2026-07-01'
          AND data_source_name != 'SPARTAN'
          AND book_date IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """,
    'ods.sfs_booked_contracts_scd (application_date)': """
        SELECT date_trunc('week', application_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM ods.sfs_booked_contracts_scd
        WHERE application_date >= '2026-07-01'
          AND current_version_flag = 1
        GROUP BY 1 ORDER BY 1
    """,
    'odsnpi.sfs_deal_detail_scd (application_received_date)': """
        SELECT date_trunc('week', application_received_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM odsnpi.sfs_deal_detail_scd
        WHERE application_received_date >= '2026-07-01'
          AND current_version_flag = 1
          AND aspect = 'application'
        GROUP BY 1 ORDER BY 1
    """,
    'sandbox.rds_rec_model_originations (con_date)': """
        SELECT date_trunc('week', con_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM sandbox.rds_rec_model_originations
        WHERE con_date >= '2026-07-01'
          AND lob = 'SFS'
        GROUP BY 1 ORDER BY 1
    """,
    'edwnpi.date_dim (max calendar_date)': """
        SELECT MAX(calendar_date)::date AS max_date,
               COUNT(*) AS total_rows
        FROM edwnpi.date_dim
    """,
}

print('=' * 80)
print('SOURCE TABLE FRESHNESS CHECK')
print('=' * 80)

table_max_dates = {}
for label, query in table_checks.items():
    print(f'\n--- {label} ---')
    try:
        df = run_diag(query)
        if 'week_start' in df.columns:
            print(df.to_string(index=False))
            max_val = df['week_start'].max()
        elif 'max_date' in df.columns:
            print(df.to_string(index=False))
            max_val = df['max_date'].iloc[0]
        else:
            print(df.to_string(index=False))
            max_val = 'N/A'
        table_max_dates[label] = str(max_val)
    except Exception as e:
        print(f'  ERROR: {e}')
        table_max_dates[label] = f'ERROR: {e}'

SOURCE TABLE FRESHNESS CHECK

--- edwnpi.los_deal_current_fact (book_date) ---
week_start  row_count
2026-06-29       1632
2026-07-06       3444
2026-07-13       3324
2026-07-20       3224
2026-07-27       3317
2026-08-03       3136
2026-08-10       3053
2026-08-17       3043
2026-08-24       3206
2026-08-31       2108

--- edwnpi.los_deal_current_fact (app_received_dtm) ---
week_start  row_count
2026-06-29       2214
2026-07-06       3236
2026-07-13       3271
2026-07-20       3264
2026-07-27       3116
2026-08-03       2957
2026-08-10       2955
2026-08-17       2742
2026-08-24       2237
2026-08-31        478

--- ods.sfs_booked_contracts_scd (application_date) ---
week_start  row_count
2026-06-29        257
2026-07-06        402
2026-07-13        435
2026-07-20        428
2026-07-27        409
2026-08-03        381
2026-08-10        349
2026-08-17        287
2026-08-24        164
2026-08-31          6

--- odsnpi.sfs_deal_detail_scd (application_received_date) ---
week_start  row_c

In [13]:
# =============================================================================
# CELL 3: ULA QUERY WEEKLY ROW COUNTS
# Wraps ste_ragu_ula.txt in a weekly count to see where rows disappear.
# Requires temp tables on the same connection.
# =============================================================================

print('Creating temp tables on diagnostic connection...')
with open('../queries/ste_ragu_temptables.txt', 'r') as f:
    conn.execute(f.read().strip())
print('Temp tables created')

ula_weekly_count_query = """
SELECT dd.calendar_week AS book_week,
       COUNT(*) AS row_count
FROM edwnpi.los_deal_current_fact AS cd
    INNER JOIN ods.sfs_booked_contracts_scd con
        ON con.sfs_application_number = cd.sfs_application_number
        AND con.current_version_flag = 1
    LEFT JOIN edwnpi.date_dim AS dd
        ON cd.book_date = dd.calendar_date
    LEFT JOIN sandbox.rds_blackbook_rollup bbvt
        ON con.sfs_application_number = bbvt.sfs_application_number
        AND bb_value_source = 'sandbox.sfs_vehicles'
        AND bb_value_type = 'history_adjusted_wholesale_avg'
    LEFT JOIN (SELECT loan_id, application_received_date, deal_detail_id,
                      source_document_id, model_score_version,
                      ROW_NUMBER() OVER (PARTITION BY source_document_id
                          ORDER BY application_received_date DESC) rn
               FROM odsnpi.sfs_deal_detail_scd
               WHERE current_version_flag = 1 AND aspect = 'application') app
        ON cd.sfs_application_number = app.source_document_id AND app.rn = 1
    LEFT JOIN odsnpi.sfs_deal_collateral_scd car
        ON app.deal_detail_id = car.deal_detail_id
        AND car.current_version_flag = 1
        AND car.collateral_role = 'Purchase'
    LEFT JOIN (SELECT collateral_id, evaluation_amt AS blackbook
               FROM odsnpi.sfs_collateral_evaluation_scd
               WHERE source = 'Blackbook'
                 AND name = 'HistoryAdjustedWholesale'
                 AND current_version_flag = 1) bb_eval
        ON car.collateral_id = bb_eval.collateral_id
    LEFT JOIN odsnpi.sfs_deal_customer_scd pb_id
        ON app.deal_detail_id = pb_id.deal_detail_id
        AND pb_id.customer_role = 'Primary'
        AND pb_id.current_version_flag = 1
    LEFT JOIN odsnpi.sfs_deal_customer_scd cb_id
        ON app.deal_detail_id = cb_id.deal_detail_id
        AND cb_id.customer_role = 'Auxiliary'
        AND cb_id.current_version_flag = 1
    LEFT JOIN (SELECT *, ROW_NUMBER() OVER (PARTITION BY deal_detail_id
                  ORDER BY created_dtm DESC) rn
               FROM odsnpi.sfs_deal_scenario_scd
               WHERE scenario_type = 'Application') dec
        ON app.deal_detail_id = dec.deal_detail_id AND dec.rn = 1
    LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
        ON dru.dealer_number = cd.dealer_number
    LEFT JOIN edwnpi.crm_dealer_dim AS cdd
        ON cdd.dealer_number = cd.dealer_number AND cdd.current_version_flag = 1
    LEFT JOIN #temp_employment_type AS ett
        ON ett.account_number = cd.account_number
    LEFT JOIN sandbox.kmx_los_new_sp lkat
        ON lkat.loan_id = cd.loan_id
    LEFT JOIN #temp_sfs_customer_credit_attributes AS ccat
        ON ccat.customer_id = pb_id.customer_id
    LEFT JOIN #temp_sfs_customer_credit_attributes AS cbcat
        ON cbcat.customer_id = cb_id.customer_id
    LEFT JOIN edwnpi.dealer_attributes_pivot AS dap
        ON cd.dealer_number = dap.dealerid AND dap.datasourceid = 17
    LEFT JOIN sandbox.loan_random_numbers lrn
        ON lrn.loan_id = cd.loan_id
    LEFT JOIN (SELECT source_document_id,
                      MIN(application_received_date)::date AS decision_date
               FROM odsnpi.sfs_deal_detail_scd
               WHERE current_version_flag = 1
               GROUP BY source_document_id) prev
        ON prev.source_document_id = cd.sfs_application_number
    LEFT JOIN (SELECT det.loan_id, det.source_document_id,
                      scen.model_score,
                      scen.total_income_amt AS con_total_income_amt,
                      det.created_dtm,
                      ROW_NUMBER() OVER (PARTITION BY det.source_document_id
                          ORDER BY det.created_dtm DESC) AS rn
               FROM sfs_deal_detail_scd det
               LEFT JOIN sfs_deal_scenario_scd scen
                   ON scen.deal_detail_id = det.deal_detail_id
               WHERE det.current_version_flag = 1
                 AND det.aspect = 'contract'
                 AND scen.scenario_type = 'Contract'
                 AND scen.current_version_flag = 1) AS model_scores
        ON model_scores.source_document_id = cd.sfs_application_number
        AND model_scores.rn = 1
    LEFT JOIN #temp_fraud AS fraud
        ON fraud.loan_id = cd.loan_id
WHERE cd.data_source_name != 'SPARTAN'
  AND cd.book_date IS NOT NULL
  AND con.application_date >= '2025-10-07'
  AND cd.book_date >= '2026-07-01'
GROUP BY 1
ORDER BY 1
"""

print('\n--- ULA (Full Query) Weekly Row Counts ---')
ula_weekly_df = run_diag(ula_weekly_count_query)
print(ula_weekly_df.to_string(index=False))

Creating temp tables on diagnostic connection...
Temp tables created

--- ULA (Full Query) Weekly Row Counts ---
 book_week  row_count
        27        813
        28        983
        29        952
        30        627
        31        456
        32        415
        33        395
        34        544
        35        891
        36        490


In [14]:
# =============================================================================
# CELL 4: RECOVERY QUERY WEEKLY ROW COUNTS
# Most likely culprit: sandbox.rds_rec_model_originations is batch-loaded.
# =============================================================================

recovery_weekly_query = """
SELECT dd.calendar_week AS book_week,
       COUNT(*) AS row_count
FROM sandbox.rds_rec_model_originations rec
    LEFT JOIN edwnpi.date_dim AS dd
        ON rec.con_date = dd.calendar_date
    LEFT JOIN edwnpi.los_deal_current_fact cd
        ON rec.account_number = cd.account_number
    LEFT JOIN (SELECT source_document_id,
                      MIN(application_received_date)::date AS decision_date
               FROM odsnpi.sfs_deal_detail_scd
               WHERE current_version_flag = 1
               GROUP BY source_document_id) prev
        ON prev.source_document_id = cd.sfs_application_number
WHERE rec.lob = 'SFS'
  AND prev.decision_date >= '2026-07-01'
GROUP BY 1
ORDER BY 1
"""

print('--- Recovery (Full Query) Weekly Row Counts ---')
recovery_weekly_df = run_diag(recovery_weekly_query)
print(recovery_weekly_df.to_string(index=False))

print('\n--- sandbox.rds_rec_model_originations MAX con_date (SFS) ---')
rec_max = run_diag("""
    SELECT MAX(con_date)::date AS max_con_date,
           COUNT(*) AS total_sfs_rows
    FROM sandbox.rds_rec_model_originations
    WHERE lob = 'SFS'
""")
print(rec_max.to_string(index=False))

--- Recovery (Full Query) Weekly Row Counts ---
 book_week  row_count
        27        210
        28        337
        29        442
        30        431
        31        439
        32        360
        33        351
        34        313
        35        223
        36         18

--- sandbox.rds_rec_model_originations MAX con_date (SFS) ---
max_con_date  total_sfs_rows
  2026-09-01           39197


In [15]:
# =============================================================================
# CELL 5: STE WEEKLY QUERY ROW COUNTS (ste_ragu_weekly.txt)
# =============================================================================

weekly_query_counts = """
WITH reports AS (
    SELECT crs.customer_id,
           attr.attribute_value::int AS raw_vantage,
           CASE WHEN raw_vantage < 300 OR raw_vantage > 850 THEN NULL
                ELSE raw_vantage END AS vantage,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY report_date DESC) rn
    FROM odsnpi.sfs_customer_report_scd crs
        LEFT JOIN odsnpi.sfs_customer_report_attribute_scd attr
            ON crs.customer_report_id = attr.customer_report_id
            AND attr.attribute_name = 'ConsumerCreditScore'
            AND attr.current_version_flag = 1
    WHERE report_type = 'Bureau'
      AND crs.current_version_flag = 1
)
SELECT date_trunc('week', cd.book_date)::date AS week_start,
       COUNT(*) AS row_count
FROM (SELECT loan_id, application_received_date, deal_detail_id,
             source_document_id,
             ROW_NUMBER() OVER (PARTITION BY source_document_id
                 ORDER BY application_received_date DESC) rn
      FROM odsnpi.sfs_deal_detail_scd
      WHERE current_version_flag = 1 AND aspect = 'application') app
    LEFT JOIN (SELECT loan_id, application_received_date, deal_detail_id,
                      source_document_id,
                      ROW_NUMBER() OVER (PARTITION BY source_document_id
                          ORDER BY created_dtm DESC) rn
               FROM odsnpi.sfs_deal_detail_scd
               WHERE current_version_flag = 1 AND aspect = 'contract') con_id
        ON app.source_document_id = con_id.source_document_id AND con_id.rn = 1
    LEFT JOIN odsnpi.sfs_deal_scenario_scd con
        ON con_id.deal_detail_id = con.deal_detail_id
        AND con.scenario_type = 'Contract'
        AND con.current_version_flag = 1
    LEFT JOIN odsnpi.sfs_deal_collateral_scd car
        ON app.deal_detail_id = car.deal_detail_id
        AND car.current_version_flag = 1
        AND car.collateral_role = 'Purchase'
    LEFT JOIN (SELECT collateral_id, evaluation_amt AS blackbook
               FROM odsnpi.sfs_collateral_evaluation_scd
               WHERE source = 'Blackbook'
                 AND name = 'HistoryAdjustedWholesale'
                 AND current_version_flag = 1) bb
        ON car.collateral_id = bb.collateral_id
    LEFT JOIN odsnpi.sfs_booked_contracts_scd cd
        ON app.source_document_id = cd.sfs_application_number
        AND cd.current_version_flag = 1
WHERE app.rn = 1
  AND cd.book_date IS NOT NULL
  AND cd.book_date >= '2026-07-01'
GROUP BY 1
ORDER BY 1
"""

print('--- STE Weekly Query Row Counts ---')
weekly_df = run_diag(weekly_query_counts)
print(weekly_df.to_string(index=False))

--- STE Weekly Query Row Counts ---
week_start  row_count
2026-06-29        316
2026-07-06        463
2026-07-13        432
2026-07-20        438
2026-07-27        419
2026-08-03        401
2026-08-10        387
2026-08-17        369
2026-08-24        416
2026-08-31        246


In [16]:
# =============================================================================
# CELL 6: JOIN-LEVEL ISOLATION (ULA)
# Progressively build the ULA query to pinpoint which join causes the drop.
# =============================================================================

isolation_queries = {
    '1. los_deal_current_fact ONLY (base)': """
        SELECT date_trunc('week', cd.book_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact AS cd
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= '2026-07-01'
        GROUP BY 1 ORDER BY 1
    """,
    '2. + INNER JOIN sfs_booked_contracts_scd': """
        SELECT date_trunc('week', cd.book_date)::date AS week_start,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact AS cd
            INNER JOIN ods.sfs_booked_contracts_scd con
                ON con.sfs_application_number = cd.sfs_application_number
                AND con.current_version_flag = 1
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND con.application_date >= '2025-10-07'
          AND cd.book_date >= '2026-07-01'
        GROUP BY 1 ORDER BY 1
    """,
    '3. + LEFT JOIN date_dim': """
        SELECT dd.calendar_week AS book_week,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact AS cd
            INNER JOIN ods.sfs_booked_contracts_scd con
                ON con.sfs_application_number = cd.sfs_application_number
                AND con.current_version_flag = 1
            LEFT JOIN edwnpi.date_dim AS dd
                ON cd.book_date = dd.calendar_date
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND con.application_date >= '2025-10-07'
          AND cd.book_date >= '2026-07-01'
        GROUP BY 1 ORDER BY 1
    """,
    '4. + LEFT JOIN sfs_deal_detail_scd (app) + prev': """
        SELECT dd.calendar_week AS book_week,
               prev.decision_date AS app_date,
               COUNT(*) AS row_count
        FROM edwnpi.los_deal_current_fact AS cd
            INNER JOIN ods.sfs_booked_contracts_scd con
                ON con.sfs_application_number = cd.sfs_application_number
                AND con.current_version_flag = 1
            LEFT JOIN edwnpi.date_dim AS dd
                ON cd.book_date = dd.calendar_date
            LEFT JOIN (SELECT source_document_id,
                              MIN(application_received_date)::date AS decision_date
                       FROM odsnpi.sfs_deal_detail_scd
                       WHERE current_version_flag = 1
                       GROUP BY source_document_id) prev
                ON prev.source_document_id = cd.sfs_application_number
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND con.application_date >= '2025-10-07'
          AND cd.book_date >= '2026-07-01'
        GROUP BY 1, 2 ORDER BY 1
    """,
}

print('=' * 80)
print('JOIN-LEVEL ISOLATION: ULA QUERY')
print('=' * 80)

for label, query in isolation_queries.items():
    print(f'\n--- {label} ---')
    try:
        df = run_diag(query)
        if 'app_date' in df.columns:
            agg = df.groupby(df.columns[0]).agg(
                row_count=('row_count', 'sum'),
                min_app_date=('app_date', 'min'),
                max_app_date=('app_date', 'max')
            ).reset_index()
            print(agg.to_string(index=False))
        else:
            print(df.to_string(index=False))
    except Exception as e:
        print(f'  ERROR: {e}')

JOIN-LEVEL ISOLATION: ULA QUERY

--- 1. los_deal_current_fact ONLY (base) ---
week_start  row_count
2026-06-29       1632
2026-07-06       3444
2026-07-13       3324
2026-07-20       3224
2026-07-27       3317
2026-08-03       3136
2026-08-10       3053
2026-08-17       3043
2026-08-24       3206
2026-08-31       2108

--- 2. + INNER JOIN sfs_booked_contracts_scd ---
week_start  row_count
2026-06-29        316
2026-07-06        463
2026-07-13        431
2026-07-20        438
2026-07-27        419
2026-08-03        401
2026-08-10        387
2026-08-17        369
2026-08-24        416
2026-08-31        246

--- 3. + LEFT JOIN date_dim ---
 book_week  row_count
        27        316
        28        463
        29        431
        30        438
        31        419
        32        401
        33        387
        34        369
        35        416
        36        246

--- 4. + LEFT JOIN sfs_deal_detail_scd (app) + prev ---
 book_week  row_count min_app_date max_app_date
        

In [17]:
# =============================================================================
# CELL 7: PYTHON-SIDE FILTER VERIFICATION
# Load cached pickle files and check max dates / weekly counts.
# =============================================================================

pickle_dir = '../../cache'
pickle_files = {
    'ula_df_total': os.path.join(pickle_dir, 'ste_ula_v1.pkl'),
    'new_recovery': os.path.join(pickle_dir, 'ste_recovery_v1.pkl'),
    'ste_weekly_raw': os.path.join(pickle_dir, 'ste_weekly_v1.pkl'),
}

print('=' * 80)
print('CACHED PICKLE FILE ANALYSIS')
print('=' * 80)

for name, path in pickle_files.items():
    print(f'\n--- {name} ({os.path.basename(path)}) ---')
    if not os.path.exists(path):
        print(f'  FILE NOT FOUND: {path}')
        continue

    mod_time = dt.datetime.fromtimestamp(os.path.getmtime(path))
    print(f'  Last modified: {mod_time:%Y-%m-%d %H:%M}')

    df = get_pickle(path)
    print(f'  Total rows: {len(df):,}')
    print(f'  Columns: {list(df.columns)}')

    for col in ['app_date', 'book_date', 'application_received_dtm', 'con_date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f'  MAX({col}): {df[col].max()}')
            print(f'  MIN({col}): {df[col].min()}')

    date_col = None
    for candidate in ['app_date', 'book_date', 'application_received_dtm']:
        if candidate in df.columns:
            date_col = candidate
            break

    if date_col:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        recent = df[df[date_col] >= '2026-07-01'].copy()
        if len(recent) > 0:
            recent['week'] = recent[date_col].dt.to_period('W-SAT')
            weekly_counts = recent.groupby('week').size().reset_index(name='row_count')
            print(f'\n  Weekly counts (>= 2026-07-01) by {date_col}:')
            print(weekly_counts.to_string(index=False))
        else:
            print(f'\n  No rows with {date_col} >= 2026-07-01')

print('\n\n--- Python end_period check ---')
today = pd.Timestamp.today().normalize()
end_period = today.to_period('W-SAT')
print(f'Today: {today.date()}')
print(f'end_period (W-SAT): {end_period}')
print(f'end_period would include dates through: {end_period.end_time.date()}')

CACHED PICKLE FILE ANALYSIS

--- ula_df_total (ste_ula_v1.pkl) ---
  Last modified: 2026-09-03 17:03
  Total rows: 82,868
  Columns: ['account_number', 'book_date', 'book_week', 'book_vintage', 'app_date', 'bbvalue', 'sale_price', 'frni_flag', 'apr', 'amt_financed', 'cash_down', 'tradein_value', 'mileage', 'purchase_type', 'model_score', 'model_score_version', 'cd_model_score', 'total_income', 'pti', 'make', 'model_year', 'vin', 'lob', 'day_of_week', 'secured_credit_card', 'chime_indicator', 'nonkmx_chime_indicator', 'cb_flag', 'fraud_adjustment', 'employment', 'specialty_dealer', 'employed_months', 'fico_score', 'vantage_score', 'state', 'pull_type', 'existing_dq_count', 'pct_auth_tradelines', 'open_tl', 'prev_co_count', 'lob_or_bucket', 'apr_bucket', 'fico_adjustment_flag', 'job_company', 'mtn_model', 'con_pti_back']
  MAX(app_date): 2026-08-31 00:00:00
  MIN(app_date): 2025-10-07 00:00:00
  MAX(book_date): 2026-09-02 00:00:00
  MIN(book_date): 2025-10-08 00:00:00

  Weekly counts (>

In [18]:
# =============================================================================
# CELL 9: PYTHON-SIDE SCORING GATE ANALYSIS
# Reproduces the postintegration notebook's processing pipeline on cached data,
# then checks each return-None gate in get_ragu_score() per vintage.
# =============================================================================

import pickle, os, warnings
import pandas as pd
import numpy as np

pickle_dir = '../../cache'

ula_raw = get_pickle(os.path.join(pickle_dir, 'ste_ula_v1.pkl'))
rec_raw = get_pickle(os.path.join(pickle_dir, 'ste_recovery_v1.pkl'))
wkly_raw = get_pickle(os.path.join(pickle_dir, 'ste_weekly_v1.pkl'))

date_col = 'app_date'
period_freq = 'W-SAT'
start_date = pd.Timestamp('2026-05-01')
end_date = pd.Timestamp.today().normalize()
start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

# --- Reproduce Cell 6: Period assignment + caps ---
ula = ula_raw[ula_raw.lob != 'Core'].copy()
ula['lob'] = 'STE'
ula['app_date'] = pd.to_datetime(ula['app_date'])
ula['book_date'] = pd.to_datetime(ula['book_date'])
ula['week'] = ula[date_col].dt.to_period(period_freq)
ula['period'] = ula['week']

mask = (ula['period'] >= start_period) & (ula['period'] <= end_period)
ula = ula[mask]

ula['bbltv'] = ula.amt_financed / ula.bbvalue.replace(0, np.nan)
ula = ula[
    (ula.bbltv <= 10.0) |
    (ula.bbvalue.isna()) |
    (ula.bbvalue == 0)
]
ula = ula[ula.pti <= 0.6]
ula = ula[ula.total_income <= 200000]

# --- Reproduce vintage string ---
ula['vintage'] = ula['period'].astype(str)

# --- Recovery ---
rec = rec_raw.copy()
rec['app_date'] = pd.to_datetime(rec['app_date'])
rec['week'] = rec[date_col].dt.to_period(period_freq)
rec['period'] = rec['week']
rec_mask = (rec['period'] >= start_period) & (rec['period'] <= end_period)
rec = rec[rec_mask]

nr = rec[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')

print('=' * 100)
print('PYTHON-SIDE SCORING GATE ANALYSIS (per vintage)')
print('=' * 100)
print(f'{"vintage":<25} {"ula_rows":>10} {"bb_pop":>10} {"bb_pct":>8} {"rec_pop":>10} {"rec_pct":>8} {"gate_hit":<20}')
print('-' * 100)

all_vintages = sorted(ula['vintage'].unique())
for vintage in all_vintages:
    vdf = ula[ula.vintage == vintage].copy()
    n_ula = len(vdf)

    if n_ula == 0:
        print(f'{vintage:<25} {0:>10} {"--":>10} {"--":>8} {"--":>10} {"--":>8} {"GATE 1: no ULA":<20}')
        continue

    vdf_sub = vdf[['account_number', date_col, 'bbvalue', 'amt_financed', 'lob', 'apr']].copy()
    mix = vdf_sub.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                        on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    bb_pop = mix[mix['bbvalue'].notna() & (mix['bbvalue'] > 0)]
    n_bb = len(bb_pop)
    bb_pct = f'{n_bb/len(mix)*100:.1f}%' if len(mix) > 0 else '--'

    rec_pop = mix[mix['bbvalue'].notna() & (mix['bbvalue'] > 0) & mix['recovery_multiplier'].notna()]
    n_rec = len(rec_pop)
    rec_pct = f'{n_rec/len(mix)*100:.1f}%' if len(mix) > 0 else '--'

    if n_bb == 0:
        gate = 'GATE 2: no bbvalue'
    elif n_rec == 0:
        gate = 'GATE 2b: no recovery'
    else:
        gate = 'PASS'

    print(f'{vintage:<25} {n_ula:>10} {n_bb:>10} {bb_pct:>8} {n_rec:>10} {rec_pct:>8} {gate:<20}')

PYTHON-SIDE SCORING GATE ANALYSIS (per vintage)
vintage                     ula_rows     bb_pop   bb_pct    rec_pop  rec_pct gate_hit            
----------------------------------------------------------------------------------------------------
2026-04-26/2026-05-02           1201        465    99.8%        444    95.3% PASS                
2026-05-03/2026-05-09           1222        495   100.0%        472    95.4% PASS                
2026-05-10/2026-05-16           1231        481    99.8%        460    95.4% PASS                
2026-05-17/2026-05-23           1324        499   100.0%        468    93.8% PASS                
2026-05-24/2026-05-30           1389        581    99.8%        541    93.0% PASS                
2026-05-31/2026-06-06           1094        469    99.6%        441    93.6% PASS                
2026-06-07/2026-06-13           1166        511    99.4%        488    94.9% PASS                
2026-06-14/2026-06-20           1097        490    99.4%        458

In [19]:
# =============================================================================
# CELL 10: DEEP DIVE - NULL COLUMN ANALYSIS FOR FAILING VINTAGES
# For each vintage that fails a gate, show which columns are NULL and
# what percentage of rows are affected.
# =============================================================================

print('=' * 100)
print('DEEP DIVE: NULL ANALYSIS FOR EACH VINTAGE (last 6 weeks)')
print('=' * 100)

key_cols = ['bbvalue', 'total_income', 'pti', 'cd_model_score', 'model_score',
            'amt_financed', 'sale_price', 'apr']

last_6 = all_vintages[-6:]
for vintage in last_6:
    print(f'\n--- {vintage} ---')

    # Pre-cap analysis (before total_income / pti filters)
    ula_precap = ula_raw[ula_raw.lob != 'Core'].copy()
    ula_precap['lob'] = 'STE'
    ula_precap['app_date'] = pd.to_datetime(ula_precap['app_date'])
    ula_precap['period'] = ula_precap['app_date'].dt.to_period(period_freq)
    ula_precap['vintage'] = ula_precap['period'].astype(str)
    vdf_precap = ula_precap[
        (ula_precap.vintage == vintage) &
        (ula_precap.period >= start_period) &
        (ula_precap.period <= end_period)
    ]
    n_precap = len(vdf_precap)

    # After total_income cap
    ti_mask = vdf_precap.total_income <= 200000
    n_after_ti = ti_mask.sum()
    n_null_ti = vdf_precap.total_income.isna().sum()

    # After pti cap
    pti_mask = vdf_precap.pti <= 0.6
    n_after_pti = pti_mask.sum()
    n_null_pti = vdf_precap.pti.isna().sum()

    print(f'  Pre-cap rows:                    {n_precap:>6}')
    print(f'  total_income NULL:               {n_null_ti:>6}  ({n_null_ti/max(n_precap,1)*100:.1f}%)')
    print(f'  Rows surviving total_income cap: {n_after_ti:>6}  (dropped {n_precap - n_after_ti - n_null_ti} by value + {n_null_ti} NULL)')
    print(f'  pti NULL:                        {n_null_pti:>6}  ({n_null_pti/max(n_precap,1)*100:.1f}%)')
    print(f'  Rows surviving pti cap:          {n_after_pti:>6}')

    # Post-cap vintage rows
    vdf = ula[(ula.vintage == vintage)].copy()
    n_postcap = len(vdf)
    print(f'  Post-all-caps rows:              {n_postcap:>6}')

    if n_postcap > 0:
        n_bb_null = vdf.bbvalue.isna().sum()
        n_bb_zero = (vdf.bbvalue == 0).sum()
        n_bb_good = ((vdf.bbvalue.notna()) & (vdf.bbvalue > 0)).sum()
        print(f'  bbvalue NULL:                    {n_bb_null:>6}  ({n_bb_null/n_postcap*100:.1f}%)')
        print(f'  bbvalue == 0:                    {n_bb_zero:>6}  ({n_bb_zero/n_postcap*100:.1f}%)')
        print(f'  bbvalue > 0:                     {n_bb_good:>6}  ({n_bb_good/n_postcap*100:.1f}%)')

        # Recovery match
        mix = vdf[['account_number']].merge(
            nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
            on='account_number', how='left'
        ).drop_duplicates(subset='account_number', keep='first')
        n_rec_null = mix.recovery_multiplier.isna().sum()
        n_rec_good = mix.recovery_multiplier.notna().sum()
        print(f'  recovery_multiplier NULL:        {n_rec_null:>6}  ({n_rec_null/len(mix)*100:.1f}%)')
        print(f'  recovery_multiplier populated:   {n_rec_good:>6}  ({n_rec_good/len(mix)*100:.1f}%)')
    else:
        print('  ** ALL ROWS ELIMINATED BY CAPS **')
        print(f'  -> Check total_income NULL rate: {n_null_ti/max(n_precap,1)*100:.1f}%')
        print(f'     (NaN <= 200000 evaluates to False in pandas, dropping NULL rows)')

DEEP DIVE: NULL ANALYSIS FOR EACH VINTAGE (last 6 weeks)

--- 2026-07-26/2026-08-01 ---
  Pre-cap rows:                       416
  total_income NULL:                    8  (1.9%)
  Rows surviving total_income cap:    408  (dropped 0 by value + 8 NULL)
  pti NULL:                             0  (0.0%)
  Rows surviving pti cap:             416
  Post-all-caps rows:                 408
  bbvalue NULL:                       152  (37.3%)
  bbvalue == 0:                         2  (0.5%)
  bbvalue > 0:                        254  (62.3%)
  recovery_multiplier NULL:            20  (5.0%)
  recovery_multiplier populated:      384  (95.0%)

--- 2026-08-02/2026-08-08 ---
  Pre-cap rows:                       412
  total_income NULL:                    4  (1.0%)
  Rows surviving total_income cap:    408  (dropped 0 by value + 4 NULL)
  pti NULL:                             0  (0.0%)
  Rows surviving pti cap:             412
  Post-all-caps rows:                 408
  bbvalue NULL:               

In [20]:
# =============================================================================
# CELL 8: SUMMARY REPORT
# =============================================================================

print('=' * 80)
print('DIAGNOSTIC SUMMARY')
print(f'Reference cutoff: week of {CUTOFF_REFERENCE}')
print('=' * 80)

summary_rows = []

for label, max_date_str in table_max_dates.items():
    try:
        max_dt = pd.Timestamp(max_date_str)
        status = 'OK' if max_dt >= pd.Timestamp(CUTOFF_REFERENCE) else 'STALE'
    except Exception:
        status = 'CHECK'
    summary_rows.append({'component': label, 'max_date': max_date_str, 'status': status})

query_results = {
    'ULA full query (cell 3)': ula_weekly_df,
    'Recovery full query (cell 4)': recovery_weekly_df,
    'STE weekly query (cell 5)': weekly_df,
}
for label, qdf in query_results.items():
    if qdf is not None and len(qdf) > 0:
        week_col = qdf.columns[0]
        max_val = str(qdf[week_col].max())
        try:
            max_dt = pd.Timestamp(max_val)
            status = 'OK' if max_dt >= pd.Timestamp(CUTOFF_REFERENCE) else 'STALE'
        except Exception:
            status = 'CHECK'
    else:
        max_val = 'NO DATA'
        status = 'FAIL'
    summary_rows.append({'component': label, 'max_date': max_val, 'status': status})

for name, path in pickle_files.items():
    if os.path.exists(path):
        df = get_pickle(path)
        for col in ['app_date', 'book_date', 'application_received_dtm']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')
                max_val = str(df[col].max().date()) if df[col].notna().any() else 'N/A'
                try:
                    status = 'OK' if pd.Timestamp(max_val) >= pd.Timestamp(CUTOFF_REFERENCE) else 'STALE'
                except Exception:
                    status = 'CHECK'
                summary_rows.append({
                    'component': f'pickle {name}.{col}',
                    'max_date': max_val,
                    'status': status
                })
                break
    else:
        summary_rows.append({'component': f'pickle {name}', 'max_date': 'MISSING', 'status': 'FAIL'})

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

stale = summary[summary['status'].isin(['STALE', 'FAIL'])]
if len(stale) > 0:
    print(f'\nSQL-SIDE BOTTLENECK(S) IDENTIFIED ({len(stale)}):')
    for _, row in stale.iterrows():
        print(f'  -> {row["component"]}  (max_date={row["max_date"]})')
else:
    print('\nAll SQL components report data past the reference cutoff.')
    print('If vintages are still missing, the issue is Python-side.')
    print('See Cells 9-10 for scoring gate analysis and NULL deep dive.')

try:
    conn.close()
    print('\nDiagnostic connection closed.')
except Exception:
    pass

DIAGNOSTIC SUMMARY
Reference cutoff: week of 2026-08-09
                                             component   max_date status
              edwnpi.los_deal_current_fact (book_date) 2026-08-31     OK
       edwnpi.los_deal_current_fact (app_received_dtm) 2026-08-31     OK
       ods.sfs_booked_contracts_scd (application_date) 2026-08-31     OK
odsnpi.sfs_deal_detail_scd (application_received_date) 2026-08-31     OK
         sandbox.rds_rec_model_originations (con_date) 2026-08-31     OK
                   edwnpi.date_dim (max calendar_date) 2030-12-31     OK
                               ULA full query (cell 3)         36  CHECK
                          Recovery full query (cell 4)         36  CHECK
                             STE weekly query (cell 5) 2026-08-31     OK
                          pickle ula_df_total.app_date 2026-08-31     OK
                          pickle new_recovery.app_date 2026-08-31     OK
                       pickle ste_weekly_raw.book_date 2026-09-02   